# logit-thermo 探索：文章の DSC ——温度をゆっくり下げて上げ、凍る温度と溶ける温度のずれ（ヒステリシス）を測る下見（Pythia-1B、GPU、L4）

- **必要な secret：なし**（`HF_TOKEN` があれば取り込みが少し速くなるだけ）。**ランタイム：L4**（ハイメモリはいりません）。見込みは 1〜1.5 時間。終わったら自動で切れます。
- もう 1 本（OLMo-2-1B）のノートブックと、別のランタイムで同時に回せます（保存するファイルの名前が違うので、ぶつかりません）。
- 探索（下見、予想なし）。1 本の生成の中で温度を 0.85 → 0.35 → 0.85（と、その逆の順）にゆっくり動かし、写しの候補への場 h（−0.3・0・＋0.3）ごとに、凍る温度と溶ける温度のずれを測る材料を集めます。h ＝ 0 だけ、2 倍の速さの往復も。記事 40 × サンプル 4。
- 計算のコード：A2-2・A2-4・A2-7・A2-7b のデータ集めのコードと `scripts/a2_explore_field.py`（`0ac42c7ab82b06d2…`）は Drive の前のノートブックから取り出してハッシュで確かめる。`scripts/a2_explore_dsc.py`（`33f47a0a3bbda183…`）はこのノートブックに埋め込み。
- 保存先 `A2_explore/dsc/`。腕は 8 本。途中で切れても、もう一度「すべて実行」すれば終わった腕は飛ばします。**結果は表示しません。**
- **通知**：🚀・腕ごとの ✅（8 回）・🎉・⚠️。名札は「logit-thermo 探索 文章の DSC（ヒステリシスの下見、Pythia-1B、温度 0.85↔0.35 の往復 × 場 3 本、L4）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo 探索 文章の DSC（ヒステリシスの下見、Pythia-1B、温度 0.85↔0.35 の往復 × 場 3 本、L4）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - explore DSC hysteresis Pythia-1B', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory < 20e9:
    name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU なし'
    notify(f'⚠️ GPU のメモリが足りません（{name}）。L4 を選んでください。ランタイムを切ります', tags='warning')
    from google.colab import runtime; runtime.unassign()
    raise SystemExit
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：計算のコード
import ast, glob, hashlib, json
def take_src(want):
    """want：{変数の名前: sha256}。Drive の logit-thermo のノートブックから、埋め込んだコードを取り出してハッシュで確かめる。"""
    got = {}
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/*.ipynb', recursive=True)):
        try:
            cells = json.load(open(p))['cells']
        except Exception:
            continue
        for c in cells:
            s = c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            if c.get('cell_type') != 'code' or 'SRC' not in s:
                continue
            try:
                body = ast.parse(s).body
            except SyntaxError:
                continue
            for n in body:
                if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id in want and n.targets[0].id not in got:
                    v = ast.literal_eval(n.value)
                    if hashlib.sha256(v.encode()).hexdigest() == want[n.targets[0].id]:
                        got[n.targets[0].id] = v
        if len(got) == len(want):
            break
    miss = [k for k in want if k not in got]
    assert not miss, ('Drive のノートブックからコードを取り出せませんでした（前のノートブックを動かしていませんか）：', miss)
    return got
_s = take_src({'SRC22': 'e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53', 'SRC24': '6ef87866ae7d895d63c6e07d48a58a2ddf4111a4802ffe98dbe7f61892c96b57', 'SRC7C': '8f8776cbe4a0e1a7b1f676c4f84ba70fb9c6c0b46ef1466969e09f0ba062b550', 'SRC7BC': 'b03ff9e2b16e2297fb9142a26e15f561caacf77216453d78e2024066c01c57d0', 'SRCF': '0ac42c7ab82b06d2e7443ba9c0c0e147c0121b5351e0b5830b1a54b46043190d'})
SRC22, SRC24, SRC7C, SRC7BC, SRCF = _s['SRC22'], _s['SRC24'], _s['SRC7C'], _s['SRC7BC'], _s['SRCF']
SRCD = '"""探索 8.33（下見）：文章の DSC ——温度をゆっくり下げて上げ、凍る温度と溶ける温度のずれ（ヒステリシス）を測る（2026-10-31 13:3x、GPU（L4）、判定しない）。\nscripts/a2_2_collect.py・a2_4_collect.py・a2_7_collect.py・a2_7b_collect.py・a2_explore_field.py を実行したあとの同じ名前空間で実行する\n（load_model・valid_mask・split_doc・load_docs・CopyIndexN・B7_N・generate_field を変えずに使う）。\n\n1 本の生成の中で、温度を歩ごとに動かす（片道でまっすぐ 0.85 ↔ 0.35）：\n  \'ch\'：冷やす → 温める（0.85 → 0.35 → 0.85）、\'hc\'：温める → 冷やす（0.35 → 0.85 → 0.35）。\n  2 つの順番は、文章が長くなるほど写せる過去が増える「位置の効き」を、ヒステリシスと切り分けるため。\n場 h（写しの候補の logit に足す、8.30 と同じ）：−0.3・0・＋0.3。片道 640 歩（\'slow\'）。h ＝ 0 だけ片道 320 歩（\'fast\'）も。\n記録：トークン、写しの候補があったか、候補を選んだか、1 歩ごとのエネルギー U ＝ −ln p（温度 1・場なし。選んだトークン）、歩ごとの温度。\n記事は 20 本ずつに分けて生成する（長い文の KV キャッシュのメモリのため）。種は腕・サンプル・分けた組ごとに決める。\n"""\nimport gc, os, time\nimport numpy as np\nimport torch\n\nD_TH, D_TL = 0.85, 0.35\nD_LEG = {\'slow\': 640, \'fast\': 320}\nD_HS = [(\'hm03\', -0.3), (\'h0\', 0.0), (\'hp03\', 0.3)]\nD_ORDERS = [\'ch\', \'hc\']\nD_ARMS = [(o, ha, h, \'slow\') for ha, h in D_HS for o in D_ORDERS] + [(o, \'h0\', 0.0, \'fast\') for o in D_ORDERS]\nD_S = 4\nD_CHUNK = 20\nD_SEED = 20261133\nD_MODELS = {\'pythia\': (\'pythia-1b\', \'EleutherAI/pythia-1b\'), \'olmo\': (\'olmo2-1b\', \'allenai/OLMo-2-0425-1B\')}\n\n\ndef d_sched(order, rate):\n    n = D_LEG[rate]\n    down, up = np.linspace(D_TH, D_TL, n), np.linspace(D_TL, D_TH, n)\n    return np.concatenate([down, up] if order == \'ch\' else [up, down])\n\n\ndef d_arm_name(order, harm, rate):\n    return f\'{order}_{harm}_{rate}\'\n\n\ndef d_seed(order, harm, rate, s, c):\n    return (D_SEED + 100000 * [\'slow\', \'fast\'].index(rate) + 10000 * D_ORDERS.index(order)\n            + 1000 * [a for a, _ in D_HS].index(harm) + 10 * s + c)\n\n\n@torch.no_grad()\ndef generate_dsc(model, prompts, Ts, h, valid, gen):\n    """Ts：歩ごとの温度。→ toks・cand・pick・U（(B, len(Ts))）。手順は generate_field と同じ（温度だけ歩ごとに変わる）。"""\n    device = valid.device\n    B = len(prompts)\n    x = torch.tensor(prompts, device=device)\n    out = model(x, use_cache=True)\n    past, logits = out.past_key_values, out.logits[:, -1, :].float()\n    idx = [CopyIndexN(p, B7_N) for p in prompts]\n    toks, cand, pick, U = [], [], [], []\n    for t in range(len(Ts)):\n        T = float(Ts[t])\n        z = logits.double().masked_fill(~valid, float(\'-inf\'))\n        lp1 = torch.log_softmax(z, dim=-1)                     # 温度 1・場なしの確率（エネルギーのものさし）\n        cands = [idx[b].candidates() for b in range(B)]\n        has = torch.tensor([len(c) > 0 for c in cands], device=device)\n        if h != 0.0:\n            rows, cols = [], []\n            for b, c in enumerate(cands):\n                rows += [b] * len(c); cols += list(c)\n            if rows:\n                z[torch.tensor(rows, device=device), torch.tensor(cols, device=device)] += h\n        p = torch.softmax(z / T, dim=-1)\n        nxt = torch.multinomial(p, 1, generator=gen)\n        u = -lp1.gather(-1, nxt)[:, 0]\n        pk = torch.tensor([int(nxt[b, 0]) in set(cands[b]) for b in range(B)], device=device)\n        for b in range(B):\n            idx[b].push(int(nxt[b, 0]))\n        toks.append(nxt[:, 0]); cand.append(has); pick.append(pk); U.append(u)\n        out = model(nxt, past_key_values=past, use_cache=True)\n        past, logits = out.past_key_values, out.logits[:, -1, :].float()\n    st = lambda v: torch.stack(v, 1).cpu().numpy()\n    return dict(toks=st(toks).astype(np.int32), cand=st(cand).astype(np.bool_), pick=st(pick).astype(np.bool_),\n                U=st(U).astype(np.float32))\n\n\ndef run_dsc(name, hf_id, docs, outdir, device, notify=print):\n    todo = [a for a in D_ARMS if not os.path.exists(f\'{outdir}/{name}@{d_arm_name(a[0], a[1], a[3])}.npz\')]\n    if not todo:\n        print(name, \'はもう全部終わっています（飛ばします）\')\n        return\n    t0 = time.time()\n    tok, model = load_model(hf_id, device)\n    assert next(model.parameters()).dtype == torch.float32\n    valid, eos = valid_mask(tok, model, device)\n    pairs = [split_doc(tok, d[\'text\']) for d in docs]\n    use = [i for i, p in enumerate(pairs) if p is not None]\n    nD = len(use)\n    P = {len(pairs[i][0]) for i in use}\n    assert len(P) == 1\n    prompts = [pairs[i][0] for i in use]\n    maxpos = getattr(model.config, \'max_position_embeddings\', None)\n    assert maxpos is None or P.pop() + 2 * D_LEG[\'slow\'] <= maxpos, (\'文が長すぎます\', maxpos)\n    chunks = [list(range(i, min(nD, i + D_CHUNK))) for i in range(0, nD, D_CHUNK)]\n    for order, harm, h, rate in todo:\n        ta = time.time()\n        Ts = d_sched(order, rate)\n        L = len(Ts)\n        R = {k: np.zeros((D_S, nD, L), dt) for k, dt in ((\'toks\', np.int32), (\'cand\', np.bool_), (\'pick\', np.bool_), (\'U\', np.float32))}\n        for s in range(D_S):\n            for c, ii in enumerate(chunks):\n                g = torch.Generator(device=device).manual_seed(d_seed(order, harm, rate, s, c))\n                r = generate_dsc(model, [prompts[i] for i in ii], Ts, h, valid, g)\n                for k in R:\n                    R[k][s, ii] = r[k]\n                torch.cuda.empty_cache()\n        arm = d_arm_name(order, harm, rate)\n        path = f\'{outdir}/{name}@{arm}.npz\'\n        np.savez_compressed(path + \'.part.npz\', Ts=Ts, h=h, order=order, rate=rate, use=np.array(use), seconds=time.time() - ta,\n                            chunk=D_CHUNK, **R)\n        os.replace(path + \'.part.npz\', path)\n        notify(f\'{name}@{arm} おわり（{time.time() - ta:.0f} 秒、通算 {time.time() - t0:.0f} 秒）\')\n    del model; gc.collect(); torch.cuda.empty_cache()\n\n\n@torch.no_grad()\ndef self_test_dsc(device):\n    """小さなランダムのモデルで：温度が一定なら generate_field と同じトークン（h ＝ 0・＋0.3）。U は −ln p（温度 1）と合う。"""\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    hf_id = \'hf-internal-testing/tiny-random-GPT2LMHeadModel\'\n    tok = AutoTokenizer.from_pretrained(hf_id)\n    model = AutoModelForCausalLM.from_pretrained(hf_id, torch_dtype=torch.float32).to(device).eval()\n    valid, _ = valid_mask(tok, model, device)\n    global CONT\n    keep = CONT\n    CONT = 60\n    try:\n        prompts = [[5, 6, 7, 8] * 4, [9, 10, 11, 12] * 4, [3, 4, 3, 4] * 4]\n        for h in (0.0, 0.3):\n            a = generate_field(model, prompts, 0.7, valid, torch.Generator(device=device).manual_seed(7), h)\n            b = generate_dsc(model, prompts, np.full(60, 0.7), h, valid, torch.Generator(device=device).manual_seed(7))\n            assert (a[\'toks\'] == b[\'toks\']).all() and (a[\'pick\'] == b[\'pick\']).all(), (\'温度一定で generate_field と違う\', h)\n        # U の確かめ：全部の列を読み直して −ln p を出す\n        x = torch.tensor([prompts[0] + b[\'toks\'][0].tolist()], device=device)\n        lg = model(x).logits[0, len(prompts[0]) - 1: len(prompts[0]) - 1 + 60].double().masked_fill(~valid, float(\'-inf\'))\n        u = -torch.log_softmax(lg, -1).gather(-1, torch.tensor(b[\'toks\'][0], device=device, dtype=torch.long)[:, None])[:, 0]\n        assert np.allclose(u.cpu().numpy(), b[\'U\'][0], atol=2e-3), np.abs(u.cpu().numpy() - b[\'U\'][0]).max()\n    finally:\n        CONT = keep\n    for o in D_ORDERS:\n        for r in (\'slow\', \'fast\'):\n            Ts = d_sched(o, r)\n            assert len(Ts) == 2 * D_LEG[r] and abs(Ts[0] - (D_TH if o == \'ch\' else D_TL)) < 1e-12 and abs(Ts[-1] - Ts[0]) < 1e-12\n            assert abs(Ts[D_LEG[r] - 1] - (D_TL if o == \'ch\' else D_TH)) < 1e-12\n    seeds = {d_seed(o, ha, r, s, c) for o, ha, _, r in D_ARMS for s in range(D_S) for c in range(2)}\n    assert len(seeds) == len(D_ARMS) * D_S * 2\n    assert len(D_ARMS) == 8\n    return \'OK\'\n'
assert hashlib.sha256(SRCD.encode()).hexdigest() == '33f47a0a3bbda18341205dedb94c6d200f295fc4136f5cb8e0ee067b2d7f518c'
exec(SRC22)
exec(SRC24)
exec(SRC7C)
exec(SRC7BC)
exec(SRCF)
exec(SRCD)
print('コードを置きました')


In [ ]:
# セル3：自己テスト → 記事（A2-2 と同じか確かめる）→ 8 本の腕 → 保存（結果は表示しない）
import json, sys, transformers
try:
    print('自己テスト：', self_test_a27('cuda'), self_test_7b('cuda'), self_test_field('cuda'), self_test_dsc('cuda'))
    os.makedirs('/content/drive/MyDrive/logit-thermo/A2_explore/dsc', exist_ok=True)
    assert SEED == 20261027, ('記事を選ぶ種が A2-2 と違います', SEED)
    docs, meta = load_docs()
    old = json.load(open('/content/drive/MyDrive/logit-thermo' + '/A2_2/docs_meta.json'))
    assert [d['text_sha256'] for d in old['docs']] == [d['text_sha256'] for d in meta['docs']], 'BBC の記事が A2-2 と違います'
    name, hf_id = D_MODELS['pythia']
    run_dsc(name, hf_id, docs, '/content/drive/MyDrive/logit-thermo/A2_explore/dsc', 'cuda', notify=lambda s: notify('✅ ' + s, tags='white_check_mark'))
    json.dump(dict(model=[name, hf_id], arms=D_ARMS, legs=D_LEG, T=[D_TH, D_TL], samples=D_S, chunk=D_CHUNK, seed=D_SEED,
                   field_sha256='0ac42c7ab82b06d2e7443ba9c0c0e147c0121b5351e0b5830b1a54b46043190d', dsc_sha256='33f47a0a3bbda18341205dedb94c6d200f295fc4136f5cb8e0ee067b2d7f518c', torch=torch.__version__, transformers=transformers.__version__,
                   python=sys.version.split()[0], gpu=torch.cuda.get_device_name(0), matmul=torch.get_float32_matmul_precision()),
              open('/content/drive/MyDrive/logit-thermo/A2_explore/dsc' + f'/run_meta_{name}.json', 'w'), ensure_ascii=False, indent=1)
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ 探索 文章の DSC（Pythia-1B）で止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：文章の DSC（Pythia-1B）の材料を A2_explore/dsc に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
